# 03 - Anomaly Visualization\n\nVisualize PatchCore anomaly maps and overlays.

In [ ]:
from pathlib import Path\nimport cv2\nimport torch\nimport matplotlib.pyplot as plt\n\nfrom app.core.patchcore import PatchcoreWrapper\nfrom app.core.anomaly_map import generate_anomaly_heatmap\nfrom app.utils.visualizer import build_annotated_image\n\nimage_path = next(Path("../data/custom/test/defective").rglob("*.png"))\nbgr = cv2.imread(str(image_path))\nrgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\nresized = cv2.resize(rgb, (224, 224))\ntensor = torch.from_numpy(resized).float().permute(2, 0, 1) / 255.0\ntensor = tensor.unsqueeze(0)\n\nmodel = PatchcoreWrapper(device="cpu")\nmodel.load_memory_bank()\nscores, maps = model.predict(tensor)\nanomaly_map = maps[0, 0]\nheatmap = generate_anomaly_heatmap(anomaly_map, image_size=(224, 224))\noverlay, _ = build_annotated_image(resized, heatmap, [], "DEFECTIVE", float(scores[0]), 0.5)\n\nfig, axes = plt.subplots(1, 3, figsize=(12, 4))\naxes[0].imshow(resized); axes[0].set_title("Input"); axes[0].axis("off")\naxes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Heatmap"); axes[1].axis("off")\naxes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")\nplt.tight_layout()